# VoltEdge MVP — ML Load Forecast Model

**Formål:** Træn en LinearRegression-model der forudsiger session-volumen per ladepunkt per time.

**Features:**
- `temperature` — fra DMI (°C)
- `wind_speed` — fra DMI (m/s)
- `spot_price` — fra Energidataservice (DKK/kWh)
- `hour_of_day` — 0–23
- `day_of_week` — 0=mandag, 6=søndag
- `historical_volume` — antal sessioner samme tidspunkt historisk

**Target:** `session_volume` — antal sessioner per ladepunkt per time

**Output:** `../load-forecast-service/ml/model.pkl`

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import requests
import sqlalchemy
from datetime import datetime, timedelta
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from dotenv import load_dotenv

load_dotenv('../.env')

MYSQL_HOST     = os.getenv('MYSQL_HOST', 'localhost')
MYSQL_PORT     = os.getenv('MYSQL_PORT', '3306')
MYSQL_USER     = os.getenv('MYSQL_USER', 'voltedge')
MYSQL_PASSWORD = os.getenv('MYSQL_PASSWORD', 'voltedge_secret')
MYSQL_SESSION_DB = os.getenv('MYSQL_SESSION_DB', 'charging_session_db')
ENERGIDATASERVICE_URL = os.getenv('ENERGIDATASERVICE_URL')
DMI_API_URL = os.getenv('DMI_API_URL')

DB_URI = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_SESSION_DB}'
print('Konfiguration indlæst')

## Trin 1 — Hent historiske sessionsdata fra charging_session_db

In [ ]:
engine = sqlalchemy.create_engine(DB_URI)

query = """
SELECT
    charger_id,
    DATE_FORMAT(session_start_time, '%Y-%m-%d %H:00:00') AS hour_bucket,
    HOUR(session_start_time)      AS hour_of_day,
    DAYOFWEEK(session_start_time) - 1 AS day_of_week,
    COUNT(*) AS session_volume
FROM charging_sessions
WHERE status = 'COMPLETED'
  AND session_start_time IS NOT NULL
GROUP BY charger_id, hour_bucket, hour_of_day, day_of_week
ORDER BY charger_id, hour_bucket
"""

try:
    sessions_df = pd.read_sql(query, engine)
    print(f'Sessionsdata: {len(sessions_df)} rækker')
    display(sessions_df.head())
except Exception as e:
    print(f'Database ikke tilgængelig: {e}')
    print('Genererer syntetiske testdata ...')

    np.random.seed(42)
    n = 500
    hours = np.random.randint(0, 24, n)
    dow   = np.random.randint(0, 7, n)
    chargers = np.random.choice(['CHR-001', 'CHR-002', 'CHR-003'], n)

    # Simuler at spidstid giver flere sessioner
    peak_boost = np.where((hours >= 7) & (hours <= 9) | (hours >= 16) & (hours <= 19), 2, 0)
    weekday_boost = np.where(dow < 5, 1, 0)
    volume = np.maximum(0, np.random.poisson(3, n) + peak_boost + weekday_boost)

    sessions_df = pd.DataFrame({
        'charger_id':     chargers,
        'hour_of_day':    hours,
        'day_of_week':    dow,
        'session_volume': volume
    })
    print(f'Syntetisk dataset: {len(sessions_df)} rækker')
    display(sessions_df.describe())

## Trin 2 — Hent historiske vejrdata fra DMI

In [ ]:
def fetch_dmi_weather(base_url):
    """Henter seneste vejrobservationer fra DMI API."""
    if not base_url:
        return None
    try:
        params = {'parameterId': 'temp_dry,wind_speed', 'limit': 100, 'sortorder': 'observed,DESC'}
        resp = requests.get(base_url, params=params, timeout=10)
        resp.raise_for_status()
        features = resp.json().get('features', [])
        records = []
        for f in features:
            props = f.get('properties', {})
            records.append({
                'parameterId': props.get('parameterId'),
                'value':       props.get('value'),
                'observed':    props.get('observed')
            })
        df = pd.DataFrame(records)
        temp_df  = df[df['parameterId'] == 'temp_dry'].rename(columns={'value': 'temperature'})
        wind_df  = df[df['parameterId'] == 'wind_speed'].rename(columns={'value': 'wind_speed'})
        weather  = pd.merge(temp_df[['observed','temperature']], wind_df[['observed','wind_speed']], on='observed')
        return weather
    except Exception as e:
        print(f'DMI API fejl: {e}')
        return None

weather_df = fetch_dmi_weather(DMI_API_URL)

if weather_df is None or weather_df.empty:
    print('Genererer syntetiske vejrdata ...')
    n = len(sessions_df)
    weather_df = pd.DataFrame({
        'temperature': np.random.normal(12, 8, n).clip(-10, 35),
        'wind_speed':  np.random.exponential(5, n).clip(0, 25)
    })
else:
    print(f'DMI vejrdata: {len(weather_df)} observationer')
    mean_temp = weather_df['temperature'].mean()
    mean_wind = weather_df['wind_speed'].mean()
    weather_df = pd.DataFrame({
        'temperature': np.random.normal(mean_temp, 5, len(sessions_df)),
        'wind_speed':  np.random.normal(mean_wind, 2, len(sessions_df)).clip(0)
    })

print(f'Vejrdata shape: {weather_df.shape}')
display(weather_df.head())

## Trin 3 — Hent historiske spotpriser fra Energidataservice

In [ ]:
def fetch_spot_prices(base_url):
    """Henter day-ahead spotpriser for DK1 (DKK/kWh)."""
    if not base_url:
        return None
    try:
        params = {'filter': '{"PriceArea":["DK1"]}', 'sort': 'TimeDK asc', 'limit': 168}  # 1 uge
        resp = requests.get(base_url, params=params, timeout=10)
        resp.raise_for_status()
        records = resp.json().get('records', [])
        df = pd.DataFrame(records)
        if not df.empty and 'SpotPriceDKK' in df.columns:
            df['spot_price'] = df['SpotPriceDKK'] / 1000.0
            return df[['TimeDK', 'spot_price']]
    except Exception as e:
        print(f'Energidataservice fejl: {e}')
    return None

spot_df = fetch_spot_prices(ENERGIDATASERVICE_URL)

if spot_df is None or spot_df.empty:
    print('Genererer syntetiske spotpriser ...')
    n = len(sessions_df)
    # Simuler typiske danske spotpriser 0.2–1.5 DKK/kWh med døgnvariation
    base_price = 0.60
    hour_var = 0.3 * np.sin(2 * np.pi * sessions_df['hour_of_day'].values / 24)
    spot_prices = np.maximum(0.05, base_price + hour_var + np.random.normal(0, 0.1, n))
else:
    mean_price = spot_df['spot_price'].mean()
    std_price  = spot_df['spot_price'].std()
    spot_prices = np.random.normal(mean_price, std_price, len(sessions_df)).clip(0)
    print(f'Spotprisdata: {len(spot_df)} timer, gennemsnit={mean_price:.4f} DKK/kWh')

sessions_df['spot_price'] = spot_prices
print('Spotpriser tilføjet til datasæt')

## Trin 4 — Kombiner til én DataFrame

In [ ]:
sessions_df = sessions_df.reset_index(drop=True)
weather_df  = weather_df.reset_index(drop=True)

# Kombiner session- og vejrdata (allerede aligned via syntetisk generering)
df = pd.concat([sessions_df, weather_df], axis=1)

# Tilføj historisk volumen som feature (lag-variabel per charger + time + dag)
df['historical_volume'] = (
    df.groupby(['charger_id', 'hour_of_day', 'day_of_week'])['session_volume']
    .transform('mean')
    .fillna(0)
    .astype(int)
)

# Drop rækker med NaN
df = df.dropna(subset=['temperature', 'wind_speed', 'spot_price', 'session_volume'])

print(f'Kombineret datasæt: {df.shape}')
print(f'Kolonner: {list(df.columns)}')
display(df[['hour_of_day', 'day_of_week', 'temperature', 'wind_speed', 'spot_price', 'historical_volume', 'session_volume']].head(10))

## Trin 5 — Træn LinearRegression model

In [ ]:
FEATURE_COLS = ['temperature', 'wind_speed', 'spot_price', 'hour_of_day', 'day_of_week', 'historical_volume']
TARGET_COL   = 'session_volume'

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print(f'Træningsdata:  {X_train.shape[0]} rækker')
print(f'Testdata:      {X_test.shape[0]} rækker')
print(f'\nKoefficienter:')
for col, coef in zip(FEATURE_COLS, model.coef_):
    print(f'  {col:25s}: {coef:+.6f}')
print(f'  {"intercept":25s}: {model.intercept_:+.6f}')

## Trin 6 — Evaluer modellen (MAE og R²)

In [ ]:
y_pred = model.predict(X_test)
y_pred_clipped = np.maximum(0, y_pred)  # session-volumen kan ikke være negativ

mae = mean_absolute_error(y_test, y_pred_clipped)
r2  = r2_score(y_test, y_pred_clipped)

print('=' * 40)
print(f'  MAE (Mean Absolute Error): {mae:.4f}')
print(f'  R²  (Forklaringsgrad):     {r2:.4f}')
print('=' * 40)

# Sammenlign forudsigelse vs. faktisk
comparison = pd.DataFrame({'faktisk': y_test, 'forudsagt': y_pred_clipped.round(2)})
display(comparison.head(10))

## Trin 7 — Gem model som model.pkl

In [ ]:
MODEL_PATH = '../load-forecast-service/ml/model.pkl'
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

with open(MODEL_PATH, 'wb') as fh:
    pickle.dump(model, fh)

file_size_kb = os.path.getsize(MODEL_PATH) / 1024
print(f'Model gemt: {MODEL_PATH} ({file_size_kb:.1f} KB)')

# Verificer at modellen kan genindlæses og lave forudsigelse
with open(MODEL_PATH, 'rb') as fh:
    loaded_model = pickle.load(fh)

test_input = np.array([[12.0, 5.0, 0.50, 8, 0, 3]])  # typisk morgen-peak
test_pred  = float(loaded_model.predict(test_input)[0])
print(f'Verifikation: forudsagt LoadIndex for morgen-peak = {max(0, test_pred):.4f}')
print('Model klar til brug af load-forecast-service.')